# wMSM synthetic multiscale reconstruction

This notebook runs the same operational workflow used in the paper on the bundled anonymized ITN-like dataset. It calibrates wMSM on macro-regions, transfers the fitted binary parameter to countries, calibrates CReMB directly on countries, and compares the two reconstructions with the binary diagnostics used in the manuscript.

In [3]:
from pathlib import Path
import sys

# Add the package source directory when the notebook is executed from notebooks/.
ROOT = Path.cwd().resolve().parent
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np

from wmsm import wmsm, cremb
from wmsm.datasets import load_synthetic_itn_like
from wmsm.metrics import link_count, reconstruction_summary

In [4]:
def print_header(title: str) -> None:
    width = 78
    print("\n" + "=" * width)
    print(title.center(width))
    print("=" * width)


def print_key_values(title: str, values: dict[str, float], keys: list[str]) -> None:
    print_header(title)
    for key in keys:
        value = values[key]
        if isinstance(value, (float, np.floating)):
            print(f"{key:>12s} : {value: .6f}")
        else:
            print(f"{key:>12s} : {value}")


def print_comparison(wmsm_summary: dict[str, float], cremb_summary: dict[str, float]) -> None:
    print_header("wMSM vs CReMB on the country layer")
    metrics = ["RE_L", "ARE_k", "MRE_k", "ARE_s", "TPR", "PPV", "TNR", "ACC"]
    print(f"{'metric':>10s} | {'wMSM':>12s} | {'CReMB':>12s} | {'difference':>12s}")
    print("-" * 56)
    for key in metrics:
        w = wmsm_summary[key]
        c = cremb_summary[key]
        print(f"{key:>10s} | {w:12.6f} | {c:12.6f} | {w-c:12.6f}")

In [5]:
data = load_synthetic_itn_like()
W_country = data["W_country"]
W_macro = data["W_macro"]
country_ids = data["country_ids"]
macro_ids = data["macro_ids"]
macro_names = data["macro_names"]

A_country = (W_country > 0).astype(float)
A_macro = (W_macro > 0).astype(float)

s_country = W_country.sum(axis=1)
s_macro = W_macro.sum(axis=1)
W_total = W_country.sum()

L_country = link_count(A_country, directed=False, include_diagonal=False)
L_macro = link_count(A_macro, directed=False, include_diagonal=True)

print_key_values(
    "Dataset summary",
    {
        "countries": float(W_country.shape[0]),
        "macroregions": float(W_macro.shape[0]),
        "L_country": L_country,
        "L_macro": L_macro,
        "W_total": W_total,
    },
    ["countries", "macroregions", "L_country", "L_macro", "W_total"],
)


                               Dataset summary                                
   countries :  36.000000
macroregions :  12.000000
   L_country :  54.000000
     L_macro :  41.000000
     W_total :  1582.000000


In [6]:
# wMSM is calibrated on macro-regions, where diagonal entries represent total
# internal trade inside each macro-region. The fitted parameter is then used on
# the country layer, where self-loops are excluded.
delta_wmsm = wmsm.fit_delta_from_strengths(
    s_macro,
    None,
    L_macro,
    directed=False,
    include_diagonal=True,
)
rho_wmsm = wmsm.rho_from_total_weight(delta_wmsm, W_total)

P_wmsm = wmsm.probability_matrix_from_strengths(
    s_country,
    None,
    delta=delta_wmsm,
    directed=False,
    include_diagonal=False,
)
Wexp_wmsm = wmsm.expected_weights_matrix_from_strengths(
    s_country,
    None,
    delta=delta_wmsm,
    rho=rho_wmsm,
    directed=False,
    include_diagonal=False,
)

summary_wmsm = reconstruction_summary(
    A_country,
    W_country,
    P_wmsm,
    Wexp_wmsm,
    directed=False,
    include_diagonal=False,
)

print_key_values(
    "wMSM calibration",
    {
        "delta": delta_wmsm,
        "rho": rho_wmsm,
        "L_exp": summary_wmsm["L_exp"],
        "L_obs": summary_wmsm["L_obs"],
    },
    ["delta", "rho", "L_exp", "L_obs"],
)


                              wMSM calibration                               
       delta :  0.000066
         rho :  0.103676
       L_exp :  55.555892
       L_obs :  54.000000


In [7]:
# CReMB is calibrated directly on the target country layer. This makes it a
# deliberately favorable fine-scale benchmark.
z_cremb = cremb.fit_delta_from_strengths(
    s_country,
    None,
    L_country,
    directed=False,
    include_diagonal=False,
)

P_cremb = cremb.probability_matrix_from_strengths(
    s_country,
    None,
    delta=z_cremb,
    directed=False,
    include_diagonal=False,
)
Wexp_cremb = cremb.expected_weights_matrix_from_strengths(
    s_country,
    None,
    directed=False,
    include_diagonal=False,
)

summary_cremb = reconstruction_summary(
    A_country,
    W_country,
    P_cremb,
    Wexp_cremb,
    directed=False,
    include_diagonal=False,
)

print_key_values(
    "CReMB calibration",
    {
        "z": z_cremb,
        "L_exp": summary_cremb["L_exp"],
        "L_obs": summary_cremb["L_obs"],
    },
    ["z", "L_exp", "L_obs"],
)


                              CReMB calibration                               
           z :  0.000074
       L_exp :  54.000000
       L_obs :  54.000000


In [8]:
print_comparison(summary_wmsm, summary_cremb)


                     wMSM vs CReMB on the country layer                      
    metric |        wMSM |        CReMB |   difference
--------------------------------------------------------
      RE_L |     0.028813 |     0.000000 |     0.028813
     ARE_k |     0.393203 |     0.367416 |     0.025787
     MRE_k |     1.190149 |     0.985359 |     0.204790
     ARE_s |     0.027778 |     0.027778 |    -0.000000
       TPR |     0.232264 |     0.210246 |     0.022018
       PPV |     0.225759 |     0.210246 |     0.015513
       TNR |     0.925324 |     0.925961 |    -0.000637
       ACC |     0.865918 |     0.864614 |     0.001305
